# 🎯 [K08] 군 규정 검색 AI - 근거 조항 안내 규정 검색 도우미 구현
### Google Colab 실습 환경 (군사법령·규정 / Level 1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K08/K08_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 군사법령·규정 (Level 1) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 20.0시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
부대 생활 규정은 분량이 많고 용어가 딱딱해, 장병이 '전기난로 써도 되나요' 같은 질문의 답을 직접 찾기 어렵습니다. 이 프로젝트에서는 가상 부대 생활 규정집을 조 단위로 정리하고, 키워드 검색에서 TF-IDF 검색, 동의어 확장으로 검색 품질을 높인 뒤, 근거 조항을 인용해 답하고 규정에 없는 질문은 거절하는 '군 규정 검색 도우미'를 구현합니다. 검색 결과를 생성형 AI에 넘기는 RAG(검색 증강 생성) 프롬프트까지 만들어, AI가 근거 없이 답을 지어내지 않게 하는 방법을 익힙니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 실습의 모든 데이터(공지, 규정, 일정 등)는 보안 규정을 준수하여 **코드 내 난수 및 가상 시나리오로 생성**됩니다.
> - 실제 부대 작전, 군사 기밀 및 개인 식별 정보는 일체 포함되지 않습니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 규정집을 조 단위로 정리하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **가상 규정집을 조 단위 검색 조각(장·조 번호·제목·본문)으로 구조화합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **규정을 장과 조로 나누어 표로 만들고 조 번호의 연속성과 조각 길이를 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `규정 조각 표, 구조 점검 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 데이터 구조화 — 검색과 근거 표시에 맞게 규정을 조 단위로 구조화했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(규정 조각 표, 구조 점검 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""가상 부대 생활 규정을 '조' 단위 검색 조각(장·조 번호·제목·본문)으로 구조화하고, 조각 길이와 구성을 점검한다."""
import re
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]


def build_chunks():
    # [TODO : 학생 실습 1] 규정집을 조 단위로 정리하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
chunks = build_chunks()
df = pd.DataFrame(chunks)
df["글자수"] = df["text"].str.len()
print(df[["no", "chapter", "title", "글자수"]].to_string(index=False))
print(f"\n장 {df['chapter'].nunique()}개, 조 {len(df)}개, 조당 평균 {df['글자수'].mean():.0f}자")
print("조 번호 연속성:", "정상" if list(df["no"]) == list(range(1, len(df) + 1)) else "누락 있음")
print("\n조 단위로 자르는 이유: 답변할 때 '제7조(휴가 신청)'처럼 근거 조항을 정확히 표시할 수 있다.")


---

## 🎯 미션 02. 평가 질문 만들기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **검색 도우미를 평가할 질문 15개와 정답 조항을 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **장병이 실제로 쓰는 표현('사진 찍어도', '전기난로')으로 질문을 만들고 정답 조항을 연결 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 2개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `평가 질문 15개와 정답 조항, 유형별 분포` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 평가 기준 설계 — 실제 표현을 반영한 평가 질문과 정답을 마련했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(평가 질문 15개와 정답 조항, 유형별 분포)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""규정 검색 도우미를 평가할 질문 15개와 정답 조항을 정리하고, 질문 유형(기한·시간·허용 여부·절차)을 분류한다."""
import re
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 2] 평가 질문 만들기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

chunks = {c["no"]: c for c in build_chunks()}


def qtype(q):
    # [TODO : 학생 실습 2] 평가 질문 만들기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rows = [{"질문": q, "유형": qtype(q), "정답 조항": chunks[a]["article"]} for q, a in QUESTIONS]
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\n유형별 질문 수:", df["유형"].value_counts().to_dict())
covered = {a for _, a in QUESTIONS}
print(f"평가 질문이 다루는 조항 {len(covered)}개 / 전체 {len(chunks)}개")
print("질문은 장병이 실제로 말하는 표현('전기난로', '사진 찍어도')으로 써서 규정 용어('전열기', '촬영')와 다르게 했다.")


---

## 🎯 미션 03. 키워드 검색 만들기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **단어 겹침으로 조항을 찾는 키워드 검색 Baseline을 만들고 정확도를 잽니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **질문과 조항에 같이 나오는 단어 수로 순위를 매기고 1위·3위 안 정확도를 계산 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `키워드 검색 함수, 정확도, 실패 질문 목록` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 기본 모델 구현 — Baseline 검색을 구현하고 정확도와 한계를 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(키워드 검색 함수, 정확도, 실패 질문 목록)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""질문과 규정 본문에 같이 나오는 단어 수로 조항을 찾는 키워드 검색 Baseline을 만들고 정확도를 잰다."""
import re
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 3] 키워드 검색 만들기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

chunks = build_chunks()


def words(t):
    # [TODO : 학생 실습 3] 키워드 검색 만들기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def keyword_search(q, k=3):
    qw = words(q)
    scored = [(len(qw & words(c["title"] + " " + c["text"])), c["no"]) for c in chunks]
    scored.sort(key=lambda x: (-x[0], x[1]))
    return [no for s, no in scored[:k] if s > 0]


rows, hit1, hit3 = [], 0, 0
for q, ans in QUESTIONS:
    res = keyword_search(q)
    h1 = bool(res) and res[0] == ans
    h3 = ans in res
    hit1 += h1; hit3 += h3
    rows.append({"질문": q[:22], "정답": f"제{ans}조", "검색 1위": f"제{res[0]}조" if res else "없음", "1위 정답": "O" if h1 else "X"})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\n키워드 검색 정확도: 1위 {hit1}/{len(QUESTIONS)} ({hit1 / len(QUESTIONS):.0%}), 3위 안 {hit3}/{len(QUESTIONS)} ({hit3 / len(QUESTIONS):.0%})")
print("한계: '신청해야' 와 '신청'처럼 어미가 붙으면 같은 단어로 보지 못한다.")


---

## 🎯 미션 04. TF-IDF 검색으로 개선하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **글자 조각 TF-IDF와 코사인 유사도로 검색을 개선합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **질문과 조항을 TF-IDF 벡터로 바꿔 유사도 순으로 정렬하고, 키워드 검색과 1위·3위 안 정확도를 비교 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `TF-IDF 검색 함수, 비교 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 검색 고도화 — TF-IDF 검색을 구현하고 Baseline 대비 개선을 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(TF-IDF 검색 함수, 비교 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""글자 조각(2~3그램) TF-IDF와 코사인 유사도로 검색을 개선하고 키워드 검색과 정확도를 비교한다."""
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 4] TF-IDF 검색으로 개선하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

chunks = build_chunks()
docs = [c["title"] + " " + c["text"] for c in chunks]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
D = vec.fit_transform(docs)


def tfidf_search(q, k=3):
    # [TODO : 학생 실습 4] TF-IDF 검색으로 개선하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def words(t):
    return set(re.findall(r"[가-힣A-Za-z0-9]{2,}", t))


def keyword_top1(q):
    sc = [(len(words(q) & words(d)), c["no"]) for d, c in zip(docs, chunks)]
    return max(sc, key=lambda x: (x[0], -x[1]))[1]


k1 = sum(keyword_top1(q) == a for q, a in QUESTIONS)
t1 = sum(tfidf_search(q)[0][0] == a for q, a in QUESTIONS)
t3 = sum(a in [n for n, _ in tfidf_search(q)] for q, a in QUESTIONS)
print(f"키워드 검색 1위 정확도 {k1 / 15:.0%}  ->  TF-IDF 검색 1위 정확도 {t1 / 15:.0%}, 3위 안 {t3 / 15:.0%}")
print("\n[검색 예시]")
for q, a in QUESTIONS[:5]:
    res = tfidf_search(q)
    print(f" Q. {q}\n    -> " + ", ".join(f"제{n}조({s})" for n, s in res) + f"  (정답 제{a}조)")


---

## 🎯 미션 05. 검색 품질 평가하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **Hit@1, Hit@3, MRR로 검색 품질을 평가하고 실패 원인을 분류합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **정답 조항의 순위로 세 지표를 계산하고, 1위를 놓친 질문을 표로 정리하고 체계화하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `검색 품질 평가표, 실패 원인 분류` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 품질 평가 — 여러 지표로 검색 품질을 평가하고 실패 원인을 찾았는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(검색 품질 평가표, 실패 원인 분류)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""Hit@1, Hit@3, MRR(정답 순위의 역수 평균)로 검색 품질을 평가하고, 틀린 질문의 원인을 분류한다."""
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 5] 검색 품질 평가하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

chunks = build_chunks()
docs = [c["title"] + " " + c["text"] for c in chunks]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(docs)
D = vec.transform(docs)


def ranks(q):
    # [TODO : 학생 실습 5] 검색 품질 평가하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rows = []
for q, a in QUESTIONS:
    r = ranks(q)
    pos = r.index(a) + 1
    rows.append({"질문": q[:24], "정답": a, "정답 순위": pos, "1위": r[0]})
df = pd.DataFrame(rows)
hit1, hit3, mrr = (df["정답 순위"] == 1).mean(), (df["정답 순위"] <= 3).mean(), (1 / df["정답 순위"]).mean()
print(f"Hit@1 {hit1:.0%} / Hit@3 {hit3:.0%} / MRR {mrr:.2f}")
print("\n[1위를 놓친 질문]")
miss = df[df["정답 순위"] > 1]
print(miss.to_string(index=False) if len(miss) else "없음")
print("\n원인 분류: 장병 표현과 규정 용어가 다른 경우(예: 사진 찍다 ↔ 촬영, 전기난로 ↔ 전열기, AI ↔ 외부 AI 서비스)")
print("-> 미션 6에서 동의어 사전으로 질문을 확장한다.")


---

## 🎯 미션 06. 동의어로 질문 넓히기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **장병 표현을 규정 용어로 바꿔 주는 동의어 사전으로 질문을 확장해 정확도를 높입니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **'사진→촬영', '전기난로→전열기' 같은 동의어 사전을 만들어 질문에 덧붙이고, 확장 전후의 Hit@1·Hit@3·MRR을 비교 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `동의어 사전, 확장 전후 비교표` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 개선·확장 — 동의어 확장으로 검색을 개선하고 부작용까지 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(동의어 사전, 확장 전후 비교표)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""장병 표현을 규정 용어로 바꿔 주는 동의어 사전으로 질문을 확장해 검색 정확도를 높이고, 확장 전후를 비교한다."""
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 6] 동의어로 질문 넓히기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

SYNONYMS = {  # 장병 표현 -> 규정 용어
    "사진": "촬영", "찍": "촬영", "전기난로": "전열기", "난로": "전열기", "ChatGPT": "외부 AI 서비스 입력",
    "AI": "외부 AI 서비스", "다친": "부상자 환자", "겨울": "동절기", "늦어질": "복귀 지연", "못 받": "연기 추가 일정",
    "고충": "고충 건의 상담", "응시료": "자기개발 지원 응시료", "몇 시": "시간",
}


def expand(q):
    # [TODO : 학생 실습 6] 동의어로 질문 넓히기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
chunks = build_chunks()
docs = [c["title"] + " " + c["text"] for c in chunks]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(docs)
D = vec.transform(docs)


def rank_of(q, a):
    s = cosine_similarity(vec.transform([q]), D).ravel()
    order = [chunks[i]["no"] for i in s.argsort()[::-1]]
    return order.index(a) + 1


rows = []
for q, a in QUESTIONS:
    rows.append({"질문": q[:22], "확장 전 순위": rank_of(q, a), "확장 후 순위": rank_of(expand(q), a)})
df = pd.DataFrame(rows)
for col in ["확장 전 순위", "확장 후 순위"]:
    print(f"{col[:5]}: Hit@1 {(df[col] == 1).mean():.0%}, Hit@3 {(df[col] <= 3).mean():.0%}, MRR {(1 / df[col]).mean():.2f}")
print("\n" + df.to_string(index=False))
print("\n확장 예시:", expand("생활관에서 전기난로를 써도 되나요?"))
print("주의: 동의어를 너무 넓게 넣으면 엉뚱한 조항이 올라온다. 추가할 때마다 평가 질문으로 다시 확인한다.")


---

## 🎯 미션 07. 근거 조항을 붙여 답하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **조항 원문을 인용해 근거가 붙은 답변을 만들고, 유사도가 낮으면 답하지 않게 합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **1위 조항 원문과 장·조 번호를 붙여 답하고, 유사도가 기준보다 낮으면 '규정에서 찾지 못함'과 담당 부서 확인을 안내 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `근거 인용 답변 함수, 답변 예시, 정확도` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 근거 기반 답변 — 근거 조항을 인용해 답하고 규정 밖 질문에는 지어내지 않게 했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(근거 인용 답변 함수, 답변 예시, 정확도)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""검색된 조항 원문을 그대로 인용해 '근거 조항이 붙은 답변'을 만들고, 검색 점수가 낮으면 답을 지어내지 않고 담당 부서 확인을 안내한다."""
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 7] 근거 조항을 붙여 답하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

# ---- 공통: 동의어 확장 + TF-IDF 검색기 ----
SYNONYMS = {
    "사진": "촬영", "찍": "촬영", "전기난로": "전열기", "난로": "전열기", "ChatGPT": "외부 AI 서비스 입력",
    "AI": "외부 AI 서비스", "다친": "부상자 환자", "겨울": "동절기", "늦어질": "복귀 지연", "못 받": "연기 추가 일정",
    "고충": "고충 건의 상담", "응시료": "자기개발 지원 응시료", "몇 시": "시간",
}


class Retriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.docs = [c["title"] + " " + c["text"] for c in chunks]
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(self.docs)
        self.D = self.vec.transform(self.docs)

    @staticmethod
    def expand(q):
        return q + " " + " ".join(v for k, v in SYNONYMS.items() if k in q)

    def search(self, q, k=3):
        s = cosine_similarity(self.vec.transform([self.expand(q)]), self.D).ravel()
        return [(self.chunks[i], float(s[i])) for i in s.argsort()[::-1][:k]]

R = Retriever(build_chunks())
THRESHOLD = 0.15   # 이보다 낮으면 '규정에서 찾지 못함'


def answer(q):
    # [TODO : 학생 실습 7] 근거 조항을 붙여 답하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
tests = ["휴가는 언제까지 신청해야 하나요?", "생활관에서 전기난로를 써도 되나요?", "ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?",
         "PX에서 라면 가격은 얼마인가요?", "전역하면 예비군은 몇 년 하나요?"]
for q in tests:
    a, no = answer(q)
    print(f"Q. {q}\nA. {a}\n")

correct = sum(answer(q)[1] == a for q, a in QUESTIONS)
print(f"평가 질문 15개 중 근거 조항까지 맞은 답변: {correct}개 ({correct / 15:.0%})")
print("원칙: 답변은 조항 원문을 인용하고, 규정에 없는 내용은 만들지 않는다.")


---

## 🎯 미션 08. 생성형 AI용 질문 묶음 만들기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **검색 결과를 넣은 생성형 AI용 질문 묶음(RAG 프롬프트)을 자동으로 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **상위 3개 조항을 [규정 발췌]로 넣고, '발췌 내용만 근거로 답하기, 근거 조항 표시, 없으면 찾을 수 없음, 숫자·기한 지어내지 않기' 규칙을 붙인 프롬프트를 생성 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `RAG 프롬프트 생성 함수, 생성 예시, 점검 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] RAG·프롬프트 설계 — 검색 결과와 규칙을 담은 프롬프트를 만들고 누락을 점검했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(RAG 프롬프트 생성 함수, 생성 예시, 점검 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""검색 결과 3개를 넣은 생성형 AI용 질문 묶음(RAG 프롬프트)을 자동으로 만들고, 프롬프트가 지켜야 할 규칙이 빠지지 않았는지 점검한다."""
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 8] 생성형 AI용 질문 묶음 만들기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

# ---- 공통: 동의어 확장 + TF-IDF 검색기 ----
SYNONYMS = {
    "사진": "촬영", "찍": "촬영", "전기난로": "전열기", "난로": "전열기", "ChatGPT": "외부 AI 서비스 입력",
    "AI": "외부 AI 서비스", "다친": "부상자 환자", "겨울": "동절기", "늦어질": "복귀 지연", "못 받": "연기 추가 일정",
    "고충": "고충 건의 상담", "응시료": "자기개발 지원 응시료", "몇 시": "시간",
}


class Retriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.docs = [c["title"] + " " + c["text"] for c in chunks]
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(self.docs)
        self.D = self.vec.transform(self.docs)

    @staticmethod
    def expand(q):
        return q + " " + " ".join(v for k, v in SYNONYMS.items() if k in q)

    def search(self, q, k=3):
        s = cosine_similarity(self.vec.transform([self.expand(q)]), self.D).ravel()
        return [(self.chunks[i], float(s[i])) for i in s.argsort()[::-1][:k]]

R = Retriever(build_chunks())

RULES = ["아래 [규정 발췌]에 있는 내용만 근거로 답한다.",
         "답변 끝에 근거 조항(예: 제7조)을 표시한다.",
         "발췌에 답이 없으면 '규정에서 찾을 수 없음'이라고 쓰고 담당 부서 확인을 안내한다.",
         "추측하거나 규정에 없는 숫자·기한을 만들지 않는다."]


def build_prompt(q, k=3):
    # [TODO : 학생 실습 8] 생성형 AI용 질문 묶음 만들기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
p = build_prompt("부대 안에서 휴대전화로 사진 찍어도 되나요?")
print(p)

checks = {"규칙 4개 포함": all(r in p for r in RULES), "근거 조항 3개 포함": p.count("제") >= 3,
          "질문 포함": "[질문]" in p, "가상 자료 표기": "교육용 가상 자료" in p}
print()
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"\n프롬프트 길이 {len(p)}자. 실제 부대 규정·문서를 외부 AI 서비스에 넣지 않으며, 이 실습은 가상 규정집으로만 한다.")


---

## 🎯 미션 09. 어려운 질문으로 시험하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **규정 밖 질문, 구어체·오타 질문으로 도우미를 시험하고 거절 기준을 정합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **유사도 기준을 여러 값으로 바꾸며 규정 질문 답변율과 규정 밖 질문 거절율을 비교하고, 두 비율이 함께 높은 기준을 비교 분석하여 최적 도구를 선정하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `기준별 답변·거절율 표, 유사도 분포 그래프, 선택한 기준` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 스트레스 시험·기준 조정 — 어려운 질문으로 시험하고 근거를 들어 거절 기준을 정했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(기준별 답변·거절율 표, 유사도 분포 그래프, 선택한 기준)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""규정에 없는 질문, 애매한 질문, 오타가 섞인 질문으로 검색 도우미를 시험해 '모르면 모른다고 답하는지'를 점검하고 임계값을 조정한다."""
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 9] 어려운 질문으로 시험하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

# ---- 공통: 동의어 확장 + TF-IDF 검색기 ----
SYNONYMS = {
    "사진": "촬영", "찍": "촬영", "전기난로": "전열기", "난로": "전열기", "ChatGPT": "외부 AI 서비스 입력",
    "AI": "외부 AI 서비스", "다친": "부상자 환자", "겨울": "동절기", "늦어질": "복귀 지연", "못 받": "연기 추가 일정",
    "고충": "고충 건의 상담", "응시료": "자기개발 지원 응시료", "몇 시": "시간",
}


class Retriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.docs = [c["title"] + " " + c["text"] for c in chunks]
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(self.docs)
        self.D = self.vec.transform(self.docs)

    @staticmethod
    def expand(q):
        return q + " " + " ".join(v for k, v in SYNONYMS.items() if k in q)

    def search(self, q, k=3):
        s = cosine_similarity(self.vec.transform([self.expand(q)]), self.D).ravel()
        return [(self.chunks[i], float(s[i])) for i in s.argsort()[::-1][:k]]

R = Retriever(build_chunks())
OUT = ["PX에서 라면 가격은 얼마인가요?", "전역하면 예비군은 몇 년 하나요?", "다음 달 사단장 방문 일정은?",
       "부대 와이파이 비밀번호 알려줘", "축구 대회 우승 상품은 뭐예요?", "월급날은 언제인가요?"]
TYPO = [("휴가 신청 언제까지 해야돼요", 7), ("비번 몇자리로 바꿔야함?", 11), ("겨울에 밖에서 훈련할때 주의할거", 14)]

in_scores = [R.search(q, 1)[0][1] for q, _ in QUESTIONS]
out_scores = [R.search(q, 1)[0][1] for q in OUT]
rows = []
for th in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
    answered_in = np.mean([s >= th for s in in_scores])
    refused_out = np.mean([s < th for s in out_scores])
    rows.append({"임계값": th, "규정 질문 답변율": f"{answered_in:.0%}", "규정 밖 질문 거절율": f"{refused_out:.0%}"})
print(pd.DataFrame(rows).to_string(index=False))

print("\n[오타·구어체 질문]")
for q, a in TYPO:
    c, s = R.search(q, 1)[0]
    print(f" {q} -> 제{c['no']}조({s:.2f}) {'정답' if c['no'] == a else '오답'}")
print("\n규정 밖 질문 유사도:", [round(s, 2) for s in out_scores])
print("판단: 임계값을 높이면 엉뚱한 답은 줄지만 정상 질문도 거절한다. 두 비율이 함께 높은 값을 고르고, 거절 시 담당 부서를 안내한다.")

plt.figure(figsize=(6, 3.5))
plt.hist(in_scores, bins=10, alpha=0.7, label="in-scope questions")
plt.hist(out_scores, bins=10, alpha=0.7, label="out-of-scope questions")
plt.xlabel("top-1 similarity"); plt.ylabel("count"); plt.legend(); plt.title("Similarity distribution (virtual regulation)")
plt.tight_layout(); plt.show()


---

## 🎯 미션 10. 규정 검색 도우미 완성하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **전체 기능을 묶은 규정 검색 도우미를 완성하고 자체 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **질문 확장, 검색, 근거 인용 답변, 거절, 질의 기록을 하나의 도우미로 통합하고, 근거 조항 정확도·거절률·처리 시간 등 자체 점검 항목을 모두 통과시킵니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 3개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `규정 검색 도우미 코드, 자체 점검 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 통합·완성 — 도우미를 통합하고 자체 점검을 통과했으며 사람의 확인 절차를 명시했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(규정 검색 도우미 코드, 자체 점검 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""규정 구조화, 질문 확장, TF-IDF 검색, 근거 인용 답변, 거절 규칙, RAG 프롬프트 생성을 하나로 묶은 '군 규정 검색 도우미'를 완성하고 자체 점검한다."""
import re
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 자료 (실제 부대·규정·인물과 무관)
# ---------------------------------------------------------------------------
REGULATIONS = [('제1장 총칙',
  [('제1조(목적)', '이 규정은 가상 한빛대대 장병의 생활 질서와 복무 기준을 정하여 안전하고 건강한 병영 생활을 보장함을 목적으로 한다.'),
   ('제2조(적용 범위)', '이 규정은 가상 한빛대대에 소속된 모든 장병과 대대에 파견된 인원에게 적용한다.')]),
 ('제2장 일과 및 생활관',
  [('제3조(일과 시간)', '기상은 06시 30분, 점호는 21시 30분, 취침은 22시로 한다. 다만 근무자와 훈련 인원은 별도 계획에 따른다.'),
   ('제4조(생활관 청결)', '생활관 청소는 매일 아침 점호 후 20분간 실시한다. 분대장은 청소 구역을 주 단위로 지정한다.'),
   ('제5조(전열기 사용)', '생활관 내 개인 전열기 사용을 금지한다. 공용 난방기 이상은 즉시 중대 행정반에 신고한다.'),
   ('제6조(개인정비 시간)', '평일 18시부터 21시까지를 개인정비 시간으로 한다. 이 시간에는 복지시설을 이용할 수 있다.')]),
 ('제3장 휴가 및 외출',
  [('제7조(휴가 신청)', '정기 휴가는 시행 7일 전까지 휴가 신청서를 작성하여 소속 분대장을 거쳐 중대장에게 제출한다.'),
   ('제8조(휴가 조정)', '같은 기간 휴가 인원이 중대 정원의 20%를 넘으면 중대장이 추첨 또는 순번으로 조정한다.'),
   ('제9조(외출)', '외출은 주말과 공휴일 09시부터 21시까지로 하며, 외출 전 행선지와 연락처를 당직 계통에 보고한다.'),
   ('제10조(복귀 지연)', '부득이한 사유로 복귀가 늦어질 경우 복귀 예정 시각 전에 당직사관에게 연락하고 사유를 보고한다.')]),
 ('제4장 보안',
  [('제11조(정보통신 보안)', '업무용 계정 비밀번호는 분기마다 변경하고 영문·숫자·특수문자를 포함해 9자리 이상으로 한다.'),
   ('제12조(촬영 금지)', '부대 안에서 개인 휴대전화로 시설·문서·화면을 촬영하거나 외부로 전송하는 행위를 금지한다.'),
   ('제13조(외부 AI 서비스)', '부대 자료와 개인정보를 외부 AI 서비스에 입력하지 않는다. 교육 목적의 사용은 가상 자료로 한정한다.')]),
 ('제5장 안전 및 보건',
  [('제14조(동절기 안전)', '동절기 야외 활동은 방한 장구를 착용하고 2인 1조로 한다. 결빙 구간에서는 차량 서행을 원칙으로 한다.'),
   ('제15조(환자 발생)', '부상자나 환자가 발생하면 즉시 당직 계통과 의무실에 알리고, 의무병의 지시에 따른다.'),
   ('제16조(예방접종)', '부대가 계획한 예방접종은 모든 인원이 받는다. 다만 의무실이 연기를 판단한 경우에는 추가 일정에 받는다.')]),
 ('제6장 복지 및 자기개발',
  [('제17조(복지시설 이용)', '복지시설은 정해진 운영 시간에 이용하며, 시설 소독·점검 시간에는 이용을 제한한다.'),
   ('제18조(자기개발 지원)', '자격증 응시료와 온라인 강좌 수강료는 연간 한도 안에서 실비로 지원한다. 신청 시 증빙 서류를 첨부한다.'),
   ('제19조(건의 및 고충)', '장병은 소원수리함, 중대장 면담, 상담관 상담을 통해 건의와 고충을 제기할 수 있으며 불이익을 받지 않는다.')])]

# ---- 공통: 규정 조각(조 단위) 만들기, 평가 질문, 검색 함수 ----
def build_chunks():
    # [TODO : 학생 실습 10] 규정 검색 도우미 완성하기 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
QUESTIONS = [  # [교육용가상자료] 평가용 질문과 정답 조항
    ("휴가는 언제까지 신청해야 하나요?", 7), ("휴가 인원이 많으면 어떻게 정하나요?", 8),
    ("주말 외출 시간은 몇 시까지인가요?", 9), ("복귀가 늦어질 것 같으면 누구에게 연락하나요?", 10),
    ("비밀번호는 몇 자리 이상이어야 하나요?", 11), ("부대 안에서 휴대전화로 사진 찍어도 되나요?", 12),
    ("ChatGPT 같은 AI에 부대 자료를 넣어도 되나요?", 13), ("겨울철 야외 활동 때 지켜야 할 것은?", 14),
    ("다친 사람이 생기면 어떻게 하나요?", 15), ("예방접종을 못 받으면 어떻게 하나요?", 16),
    ("체력단련실을 못 쓰는 시간이 있나요?", 17), ("자격증 응시료를 지원받을 수 있나요?", 18),
    ("고충이 있으면 어디에 말하나요?", 19), ("점호는 몇 시에 하나요?", 3), ("생활관에서 전기난로를 써도 되나요?", 5),
]

# ---- 공통: 동의어 확장 + TF-IDF 검색기 ----
SYNONYMS = {
    "사진": "촬영", "찍": "촬영", "전기난로": "전열기", "난로": "전열기", "ChatGPT": "외부 AI 서비스 입력",
    "AI": "외부 AI 서비스", "다친": "부상자 환자", "겨울": "동절기", "늦어질": "복귀 지연", "못 받": "연기 추가 일정",
    "고충": "고충 건의 상담", "응시료": "자기개발 지원 응시료", "몇 시": "시간",
}


class Retriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.docs = [c["title"] + " " + c["text"] for c in chunks]
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit(self.docs)
        self.D = self.vec.transform(self.docs)

    @staticmethod
    def expand(q):
        return q + " " + " ".join(v for k, v in SYNONYMS.items() if k in q)

    def search(self, q, k=3):
        s = cosine_similarity(self.vec.transform([self.expand(q)]), self.D).ravel()
        return [(self.chunks[i], float(s[i])) for i in s.argsort()[::-1][:k]]


class RegulationAssistant:
    def __init__(self, threshold=0.15):   # 미션 9에서 정한 임계값
        self.R = Retriever(build_chunks())
        self.threshold = threshold
        self.log = []

    def ask(self, q):
        hits = self.R.search(q, 3)
        top, score = hits[0]
        if score < self.threshold:
            ans = "규정에서 관련 조항을 찾지 못했습니다. 행정반 또는 담당 부서에 확인하세요."
            ref = None
        else:
            ans = f"{top['text']} (근거: {top['article']})"
            ref = top["no"]
        self.log.append({"q": q, "ref": ref, "score": round(score, 3)})
        return ans, ref, [c["article"] for c, _ in hits]


A = RegulationAssistant()
for q in ["외출하고 늦게 들어올 것 같아요", "자격증 응시료 지원돼요?", "PX 라면 가격은?"]:
    ans, ref, cand = A.ask(q)
    print(f"Q. {q}\nA. {ans}\n   후보: {', '.join(cand)}\n")

t0 = time.perf_counter()
res = [A.ask(q)[1] for q, _ in QUESTIONS]
ms = (time.perf_counter() - t0) / len(QUESTIONS) * 1000
acc = np.mean([r == a for r, (_, a) in zip(res, QUESTIONS)])
out = ["PX에서 라면 가격은 얼마인가요?", "다음 달 사단장 방문 일정은?", "월급날은 언제인가요?"]
refused = np.mean([A.ask(q)[1] is None for q in out])
checks = {"규정 질문 근거 조항 정확도 80% 이상": acc >= 0.8, "규정 밖 질문 거절 2/3 이상": refused >= 0.66,
          "답변한 질문은 모두 유사도 임계값 이상": all(l["score"] >= A.threshold for l in A.log if l["ref"] is not None),
          "질문당 처리 100ms 이내": ms < 100, "질의 기록 저장": len(A.log) >= 15}
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {sum(checks.values())}/{len(checks)} 통과 (정확도 {acc:.0%}, 거절률 {refused:.0%}, 질문당 {ms:.1f} ms)")
print("\n※ 이 도우미는 가상 규정집 학습용이다. 실제 규정 해석과 적용은 지휘계통과 담당 부서의 확인을 따른다.")
